In [ ]:
import spectral as sp
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import scipy.io
import sys
import os
import pandas as pd

sys.path.append('../')
from preprocessing import *
%matplotlib inline

#### Ws
Wavelengths: wavelengths \
White Image: refIntensity \
hyperspectral image: spectralDataAll (1st dim bands, 2nd dim time)


#### DarkCount
Dark Image: DarkCount


In [ ]:
data_folder = "../../datasets/piglet/lwp475"
class_labels = ["Not labled", "Normal", "Tumor", "Hypervasculized", "Background"]

# Load the sp data
img_mat = scipy.io.loadmat('../../datasets/piglet/lwp475/LWP475_Ws_14Nov2016_1.mat')
img_darkcount = scipy.io.loadmat('../../datasets/piglet/lwp475/LWP475_DarkCount_14Nov2016.mat')

wavelengths = img_mat['wavelengths'].astype(float)
white_full = img_mat['refIntensity'].astype(float)
dark_full = img_darkcount['DarkCount'].astype(float)
spectr = img_mat['spectralDataAll'].astype(float)

print(spectr.shape)
print(wavelengths.shape)


### plot second derivative

In [ ]:
from numpy.polynomial.polynomial import Polynomial

def second_derivative(bands, extinction, window_length=10, polyorder=2, ax=None):
    """
    calculates the second derivative of the spectra
    input:  x, y, bands and values of the spectra
            window_length, length of the filter window
            polyorder, order of the polynomial
    output: x, d2y, bands and second derivative of the spectra
    """
    d2y_fit = []
    new_bands = []
    for i in range(window_length//2,len(bands)-window_length//2):
        x = bands[i-window_length//2:i+window_length//2]
        y = extinction[i-window_length//2:i+window_length//2]
        c = Polynomial.fit(x, y, polyorder)
        # if ax is not None and i%40==0:
        #     ax.plot(x, c(x), '--',)
        # d2y = c.deriv(2)(x)[window_length//2]
        # d2y_fit.append(d2y)
        # get coefficients of the polynomial
        c = c.convert().coef
        d2y_fit.append(c[2]*(bands[i]**2))
        new_bands.append(bands[i])
    return np.array(d2y_fit), np.array(new_bands)

In [ ]:
white_ref = np.mean(white_full, axis=-1)[np.newaxis, :]
dark_ref = np.mean(dark_full, axis=-1)[np.newaxis, :]

# extract wavelengths between 760 and 1000
band_limits = [760, 1000]
band_centers = np.asarray(img_mat['wavelengths'].astype(float)).squeeze()
spectr_cropped, _ = band_removal(spectr.T, band_limits, band_centers)
white_ref_cropped, _ = band_removal(white_ref, band_limits, band_centers)
white_ref_cropped = smooth_spectral(white_ref_cropped, 20)
dark_ref_cropped, band_centers_new = band_removal(dark_ref, band_limits, band_centers)

spectr_calib = calibrate_img(spectr_cropped, white_ref_cropped, dark_ref_cropped)
spectr_smooth = smooth_spectral(spectr_calib, 20)

fix, ax = plt.subplots(1, 1, figsize=(10, 10))
fig_deriv, ax_deriv = plt.subplots(1, 1, figsize=(10, 10))
ax_deriv.set_title("Second derivative of the calibrated spectra")
for i in range(73,220,30):
    ax.plot(band_centers_new, spectr_smooth[i], label="Spectrum {}".format(i))
    spectr_d2y, bands_d2y = second_derivative(band_centers_new, spectr_smooth[i], window_length=20, ax=ax)
    ax_deriv.plot(bands_d2y, spectr_d2y)
ax.plot(band_centers_new, white_ref_cropped.T, 'r',linewidth=2, label="White reference")
ax.legend()
ax.set_title("Raw spectra and white reference")


### apply FFT to spectra

In [ ]:
import pywt

white_ref = np.mean(white_full, axis=-1)[np.newaxis, :]
dark_ref = np.mean(dark_full, axis=-1)[np.newaxis, :]

# extract wavelengths between 760 and 1000
band_limits = [760, 1000]
band_centers = np.asarray(img_mat['wavelengths'].astype(float)).squeeze()
spectr_cropped, _ = band_removal(spectr.T, band_limits, band_centers)
white_ref_cropped, _ = band_removal(white_ref, band_limits, band_centers)
white_ref_cropped = smooth_spectral(white_ref_cropped, 20)
dark_ref_cropped, band_centers_new = band_removal(dark_ref, band_limits, band_centers)

spectr_calib = calibrate_img(spectr_cropped, white_ref_cropped, dark_ref_cropped)
spectr_smooth = smooth_spectral(spectr_calib, 20)

fig1, ax1 = plt.subplots(1, 1, figsize=(10, 10))
fig2, ax2 = plt.subplots(1, 1, figsize=(10, 10))
fig3, ax3 = plt.subplots(1, 1, figsize=(10, 10))
fig4, ax4 = plt.subplots(1, 1, figsize=(10, 10))
fig5, ax5 = plt.subplots(1, 1, figsize=(10, 10))

white_ref_f = np.fft.fft(np.squeeze(white_ref_cropped))
white_ref_f[20:-20] = 0
white_ref_filtered = np.abs(np.fft.ifft(white_ref_f))
ax2.plot(band_centers_new, np.abs(white_ref_filtered))

dark_ref_f = np.fft.fft(np.squeeze(dark_ref_cropped))
ax1.plot(dark_ref_f)
ax1.set_ylim(-10000,10000)

# for i in range(73,220,30):
    # sp = spectr_cropped[i]

    # # apply FFT to the spectrum
    # f = np.fft.fft(sp)
    # # plot the frequency spectrum
    # ax1.plot(band_centers_new, np.abs(f))
    # ax1.set_yscale('log')
    # # pass the frequency spectrum through a low pass filter
    # f[10:] = 0
    # # reconstruct the signal
    # sp_filtered = np.fft.ifft(f)
    # ax2.plot(band_centers_new, np.abs(sp_filtered))
    # ax2.plot(band_centers_new, sp)

    # # calibrate spectrum
    # sp_calib = calibrate_img(np.abs(sp_filtered), white_ref_filtered, np.zeros_like(dark_ref_cropped))
    # ax3.plot(band_centers_new, np.squeeze(sp_calib))

    # # apply wavelet transform to the spectrum
    # cA, cD = pywt.dwt(sp, 'db2')
    # ax4.plot(cD)
    # cD[5:] = 0

    # sp_filtered = pywt.idwt(cA, cD, 'db2')
    # ax5.plot(band_centers_new, sp_filtered)
    

In [ ]:
def plot_spectrum(spectr, wavelengths):
    N,T = spectr.shape

    fig, ax = plt.subplots()
    colors = matplotlib.cm.viridis(np.linspace(0, 1, T))
    for i in range(T):
        ax.plot(wavelengths, spectr[:, i], color=colors[i])
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Intensity')
    # Create a ScalarMappable for the colorbar
    sm = plt.cm.ScalarMappable(cmap=matplotlib.cm.viridis, norm=plt.Normalize(0, T - 1))
    sm.set_array(np.arange(0, T))  # Specify the array directly
    # Add colorbar using the ScalarMappable
    cbar = plt.colorbar(sm, ax=ax, ticks=np.arange(0, T, 1000))
    cbar.set_label('Time')

    return fig, ax


### Plot all raw spectra from the dataset

In [ ]:
# iterate though all directories in ../datasets/piglet
dataset_path = "../../datasets/piglet/"
for pig in os.listdir(dataset_path):
    # iterate though all files in the directory and check if they are .mat files and contain ws in the filename
    if not os.path.isdir(dataset_path + pig):
        continue
    for file in os.listdir(dataset_path + pig):
        # print(file)
        if "_Ws" in file and file.endswith(".mat")::
            # load the data
            print("Loading " + file)
            img_mat = scipy.io.loadmat(dataset_path + pig + "/" + file)
            wavelengths = img_mat['wavelengths'].astype(float)
            spectr = img_mat['spectralDataAll'].astype(float)

            # plot the spectrum
            fig, ax = plot_spectrum(spectr, wavelengths)
            fig.savefig(dataset_path + pig + "/" + file.split(".mat")[0] + ".png")

### Plot all white and dark references and concentration changes

In [ ]:
xlsx_path = "overview.xlsx"
df = pd.read_excel(xlsx_path, sheet_name="Sheet1", engine="openpyxl")

fig_dark, ax_dark = plt.subplots(10,5, figsize=(20,40))
fig_white, ax_white = plt.subplots(10,5, figsize=(20,40))
fig_concentration, ax_concentration = plt.subplots(10,5, figsize=(20,40))

dark_imgs = []
for i in range(len(df)):
    path = f"../../datasets/piglet/{df.piglet[i]}"
    ax_dark[i//5, i%5].set_title(df.darkCount[i], fontsize=12)
    ax_white[i//5, i%5].set_title(df.spectrum[i], fontsize=12)
    ax_concentration[i//5, i%5].set_title(df.spectrum[i], fontsize=12)

    img_mat = scipy.io.loadmat(path + "/" + df.spectrum[i])
    white_img = img_mat['refIntensity'].astype(float)
    white_img = np.mean(white_img, axis=1)
    ax_white[i//5, i%5].plot(wavelengths, white_img)
    ax_white[i//5, i%5].set_ylim([0, 10000])

    # check if the concentration is in the file
    if "Results" in img_mat:
        concentration = img_mat["Results"]["AllConcentration"][0][0][0][0]
        ax_concentration[i//5, i%5].plot(concentration)
        ax_concentration[i//5, i%5].set_ylim([-20,20])

    filename = str(df.darkCount[i]).split(",")[0]
    if not filename.startswith("LWP"):
        continue
    dark_img = scipy.io.loadmat(path + "/" + filename)['DarkCount'].astype(float)
    dark_img = np.mean(dark_img, axis=1)
    ax_dark[i//5, i%5].plot(wavelengths, dark_img)
    ax_dark[i//5, i%5].set_ylim([0, 10000])

    if df.piglet[i] != "LWP511":
        dark_imgs.append(dark_img)



# plot mean and std as area of dark images
plt.figure()
dark_imgs = np.array(dark_imgs)
dark_imgs_mean = np.mean(dark_imgs, axis=0)
dark_imgs_std = np.std(dark_imgs, axis=0)
plt.plot(wavelengths, dark_imgs_mean, label="DarkCount mean")
plt.fill_between(wavelengths.squeeze(), dark_imgs_mean - 10*dark_imgs_std, dark_imgs_mean + 10*dark_imgs_std, alpha=0.5, label="10*std")
plt.legend()


In [ ]:
xlsx_path = "overview.xlsx"
df = pd.read_excel(xlsx_path, sheet_name="Sheet1", engine="openpyxl")

dataset_path = "../../datasets/piglet/"
for i in range(len(df)):

    if df.good[i] == "No" or df.good[i] == "?":
        continue

    piglet = df.piglet[i]

    img_mat = scipy.io.loadmat(dataset_path + piglet + "/" + df.spectrum[i])
    wavelengths = img_mat['wavelengths'].astype(float)
    white_img = img_mat['refIntensity'].astype(float)
    spectr = img_mat['spectralDataAll'].astype(float)
    dark_img = scipy.io.loadmat(dataset_path + piglet + "/" + df.darkCount[i])["DarkCount"].astype(float)

    # select wavelengths between 760 and 900
    idx = (wavelengths >= 760) & (wavelengths <= 900)
    wavelengths_cut = wavelengths[idx]
    spectr_cut = spectr[idx.squeeze()]
    white_img_cut = white_img[idx.squeeze()]
    dark_img_cut = dark_img[idx.squeeze()]

    # normalize the spectrum
    spectr_cut = (spectr_cut - dark_img_cut[:, 0, None]) / (white_img_cut[:, 0, None] - dark_img_cut[:, 0, None])

    # plot the spectrum
    fig, ax = plt.subplots(1,3, figsize=(20,5))
    plt.suptitle(df.spectrum[i])
    ax[0].plot(wavelengths_cut, dark_img_cut[:, 0, None])
    ax[0].set_title("Dark image")
    ax[1].plot(wavelengths_cut, white_img_cut[:, 0, None])
    ax[1].set_title("White image")
    ax[2].plot(wavelengths_cut, white_img_cut[:, 0, None] - dark_img_cut[:, 0, None])
    ax[2].set_title("White image - Dark image")
    fig, ax = plot_spectrum(spectr_cut[:,0:1000:10], wavelengths_cut)
    ax.set_title(f"{df.spectrum[i]} normalized spectrum (every 100th time step)", fontsize=12)
    # ax.set_ylim([0, 8])
    # fig.savefig(dataset_path + pig + "/" + file.split(".mat")[0] + "_normalized.png")

In [ ]:
xlsx_path = "overview.xlsx"
df = pd.read_excel(xlsx_path, sheet_name="Sheet1", engine="openpyxl")

dataset_path = "../../datasets/piglet/"
for i in range(len(df)):

    if df.good[i] != "Yes":
        continue

    piglet = df.piglet[i]

    img_mat = scipy.io.loadmat(dataset_path + piglet + "/" + df.spectrum[i])
    wavelengths = img_mat['wavelengths'].astype(float)
    spectr = img_mat['spectralDataAll'].astype(float)
    concentrations = img_mat['allConcent'].astype(float)

    # select wavelengths between 760 and 900
    idx = (wavelengths >= 760) & (wavelengths <= 900)
    wavelengths_cut = wavelengths[idx]
    spectr_cut = spectr[idx.squeeze()]

    # plot the spectrum
    fig, ax = plot_spectrum(spectr_cut[:,0:1000:10], wavelengths_cut)
    ax.set_title(f"{df.spectrum[i]} spectrum (every 10th time step)", fontsize=12)
    fig, ax = plt.subplots()
    ax.plot(concentrations.T[:1000,:4], label=["HHb","HbO2","oxCCo","HbT"])
    ax.set_title(f"{df.spectrum[i]} concentrations", fontsize=12)
    ax.legend()